# 第7章 资产定价与因子模型 —— 配套代码

对应正文 `book/part2/07-factor-models.md`。

> **运行前准备**：请先在终端执行 `uv run python scripts/make_sample_data.py` 生成内置示例数据。

## 数据说明

本 notebook 使用两类数据：

1. **内置价格数据**（`load_sample_prices()`）：合成的 4 只 A 股风格资产（BANK/LIQUOR/TECH/UTILITY），约 750 个交易日，用于 CAPM 时序回归演示，**完全离线可跑**。
2. **教学示意因子数据**：由于内置数据无基本面信息，无法构造真实 SMB/HML，多因子部分使用固定随机种子（`np.random.default_rng(42)`）生成的**模拟因子序列**。**这些数据仅用于演示多因子回归的操作步骤与结果解读，不代表真实 A 股因子结论**。

## 演示内容

1. 环境初始化与数据加载
2. 构造市场代理组合与超额收益
3. 单因子 CAPM 回归（OLS 摘要解读）
4. 四只股票 Beta 对比汇总
5. SML 可视化 + 散点拟合图
6. Beta 条形图 + R2 方差分解
7. （示意数据）生成模拟因子序列
8. （示意数据）FF 三因子回归
9. （示意数据）各股票 FF3 汇总
10. （示意数据）Carhart 四因子回归
11. 因子相关性与 VIF 多重共线性诊断
12. 业绩归因分解（CAPM，真实数据）
13. 因子累计收益时序图
14. 习题参考解答：习题 7.1
15. 习题参考解答：习题 7.2
16. 习题参考解答：习题 7.3
17. 习题参考解答：习题 7.4

In [ ]:
# Cell 1：环境初始化与数据加载
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import statsmodels.api as sm
from scipy import stats

from fds import load_sample_prices, daily_returns, set_chinese_font

set_chinese_font()

prices = load_sample_prices()
rets   = daily_returns(prices)       # 简单日收益率

TICKERS      = list(rets.columns)    # ['BANK', 'LIQUOR', 'TECH', 'UTILITY']
RF_ANNUAL    = 0.02                  # 年化无风险利率 2%
RF_DAILY     = RF_ANNUAL / 252       # 折日无风险利率
TRADING_DAYS = 252

print(f'价格数据维度：{prices.shape}')
print(f'收益率数据维度：{rets.shape}')
print(f'交易日范围：{rets.index[0].date()} 至 {rets.index[-1].date()}')
print(f'无风险利率（年化）：{RF_ANNUAL:.1%}，折日：{RF_DAILY:.6f}')
prices.tail(3)

## 7.2 构造市场代理组合与超额收益

由于内置数据只有 4 只股票，我们用**等权组合**作为市场组合的代理。

CAPM 时序回归方程：

$$r_{i,t} - r_{f,t} = \alpha_i + \beta_i (r_{m,t} - r_{f,t}) + \varepsilon_{i,t}$$

其中 $r_{f,t}$ 为日度无风险利率（年化 2% ÷ 252），$r_{m,t}$ 为等权市场组合日收益率。

In [ ]:
# Cell 2：构造市场代理与超额收益序列

market_ret = rets.mean(axis=1)       # 等权市场组合
market_ret.name = 'Market'
mkt_excess  = market_ret - RF_DAILY  # 市场超额收益
excess_rets = rets.sub(RF_DAILY)     # 各股票超额收益

print('=== 市场代理组合统计 ===')
print(f'日均收益：{market_ret.mean():.4f}')
print(f'日波动率：{market_ret.std():.4f}')
print(f'年化收益：{market_ret.mean() * TRADING_DAYS:.2%}')
print(f'年化波动：{market_ret.std() * np.sqrt(TRADING_DAYS):.2%}')
print(f'市场风险溢价（年化）：{mkt_excess.mean() * TRADING_DAYS:.2%}')
print()
print('各股票日超额收益（前 3 行）')
excess_rets.head(3)

## 7.3 单因子 CAPM 回归：以 TECH 为例

用 `statsmodels.OLS` 估计 CAPM，重点解读：
- `const`（截距）= $\alpha$，CAPM 下应为 0
- 斜率 = $\beta$，衡量市场敏感度
- $R^2$ = 系统性风险占总风险比例
- $t$-统计量和 $p$-值：检验系数是否显著异于 0

In [ ]:
# Cell 3：TECH 单因子 CAPM 回归（详细摘要）

y = excess_rets['TECH']
X = sm.add_constant(mkt_excess)
X.columns = ['alpha_const', 'MKT']

result_tech = sm.OLS(y, X).fit()
print(result_tech.summary())

In [ ]:
# Cell 4：四只股票 CAPM 回归汇总

capm_results = {}
X_mkt = sm.add_constant(mkt_excess)
X_mkt.columns = ['alpha_const', 'MKT']

rows = []
for ticker in TICKERS:
    res = sm.OLS(excess_rets[ticker], X_mkt).fit()
    capm_results[ticker] = res
    rows.append({
        '股票': ticker,
        'Alpha年化': round(res.params['alpha_const'] * TRADING_DAYS, 4),
        'Alpha_t': round(res.tvalues['alpha_const'], 3),
        'Alpha_p': round(res.pvalues['alpha_const'], 4),
        'Beta': round(res.params['MKT'], 4),
        'Beta_t': round(res.tvalues['MKT'], 3),
        'R2': round(res.rsquared, 4),
        '特质风险': round(1 - res.rsquared, 4),
    })

df_capm = pd.DataFrame(rows).set_index('股票')
print('=== 四只股票 CAPM 回归结果汇总 ===')
print(df_capm.to_string())
print()
print('Alpha_p < 0.05：在 5% 水平下 alpha 显著异于 0（CAPM 预测为 0）')
print('Beta_t 大：市场因子对该股票的解释力显著')

## 7.4 CAPM 可视化：SML 与散点图

**证券市场线（SML）**：横轴为 $\beta$，纵轴为年化超额收益。
CAPM 预测所有资产都应落在 SML 上；实际位置偏离代表正/负 alpha。

In [ ]:
# Cell 5：SML 可视化 + TECH 散点回归图

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

# -- 左图：SML --
ax1 = axes[0]
mkt_premium_ann   = mkt_excess.mean() * TRADING_DAYS
betas             = {t: capm_results[t].params['MKT'] for t in TICKERS}
actual_excess_ann = excess_rets.mean() * TRADING_DAYS

beta_range = np.linspace(0, max(betas.values()) * 1.25, 100)
ax1.plot(beta_range, beta_range * mkt_premium_ann, 'k--', lw=2,
         label=f'SML（市场溢价={mkt_premium_ann:.1%}/年）')

for i, ticker in enumerate(TICKERS):
    beta_i    = betas[ticker]
    capm_pred = beta_i * mkt_premium_ann
    actual    = actual_excess_ann[ticker]
    ax1.annotate('', xy=(beta_i, actual), xytext=(beta_i, capm_pred),
                 arrowprops=dict(arrowstyle='->', color=colors[i], lw=1.5))
    ax1.scatter(beta_i, actual, color=colors[i], s=100, zorder=5, label=ticker)
    alpha_ann = capm_results[ticker].params['alpha_const'] * TRADING_DAYS
    ax1.annotate(f'{ticker}\na={alpha_ann:.1%}',
                 xy=(beta_i, actual), xytext=(8, 5),
                 textcoords='offset points', color=colors[i], fontsize=9)

ax1.axhline(0, color='gray', lw=0.8, linestyle=':')
ax1.set_xlabel('Beta (β)')
ax1.set_ylabel('年化超额收益')
ax1.set_title('证券市场线（SML）与实际收益对比')
ax1.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax1.legend(fontsize=9)

# -- 右图：TECH 散点 + 回归线 --
ax2 = axes[1]
res_t  = capm_results['TECH']
x_vals = mkt_excess.values
y_vals = excess_rets['TECH'].values
ax2.scatter(x_vals, y_vals, alpha=0.3, s=15, color='#2ca02c', label='日度数据点')

x_line = np.linspace(x_vals.min(), x_vals.max(), 100)
y_line = res_t.params['alpha_const'] + res_t.params['MKT'] * x_line
b_str  = f"{res_t.params['MKT']:.2f}"
a_str  = f"{res_t.params['alpha_const']*TRADING_DAYS:.1%}"
ax2.plot(x_line, y_line, 'r-', lw=2,
         label=f'CAPM fit: a={a_str}/年, b={b_str}')

ax2.axhline(0, color='gray', lw=0.8, linestyle=':')
ax2.axvline(0, color='gray', lw=0.8, linestyle=':')
ax2.set_xlabel('市场超额收益（日）')
ax2.set_ylabel('TECH 超额收益（日）')
ax2.set_title(f'TECH CAPM 散点图（R2={res_t.rsquared:.3f}）')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()
print('Beta 汇总：')
for t in TICKERS:
    print(f'  {t}: beta = {betas[t]:.4f}')

In [ ]:
# Cell 6：Beta 条形图 + R2 方差分解

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

# 左图：Beta 对比
ax1 = axes[0]
beta_vals = [capm_results[t].params['MKT'] for t in TICKERS]
bars = ax1.bar(TICKERS, beta_vals, color=colors, alpha=0.8, edgecolor='white', linewidth=1.5)
ax1.axhline(1.0, color='black', lw=1.5, linestyle='--', label='beta=1（市场基准）')
ax1.set_title('各股票 Beta（市场敏感度）对比')
ax1.set_ylabel('Beta')
ax1.set_ylim(0, max(beta_vals) * 1.25)
ax1.legend()
for bar, val in zip(bars, beta_vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# 右图：R2 分解
ax2 = axes[1]
r2_vals   = [capm_results[t].rsquared for t in TICKERS]
idio_vals = [1 - r for r in r2_vals]
x = np.arange(len(TICKERS))
width = 0.5
bar1 = ax2.bar(x, r2_vals,   width, label='系统性风险 (R2)',  color='steelblue', alpha=0.85)
       
ax2.bar(x, idio_vals, width, bottom=r2_vals,
        label='特质风险 (1-R2)', color='salmon', alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels(TICKERS)
ax2.set_title('方差分解：系统性风险 vs 特质风险')
ax2.set_ylabel('占总方差比例')
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax2.set_ylim(0, 1.0)
ax2.legend()
for bar, val in zip(bar1, r2_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, val/2,
             f'{val:.1%}', ha='center', va='center',
             fontsize=10, color='white', fontweight='bold')

plt.tight_layout()
plt.show()
print('R2 越高，CAPM 对该股票解释力越强；1-R2 越高，特质风险越大')

## 7.5 多因子回归：Fama-French 三因子（教学示意数据）

> **重要提示**：以下使用 `np.random.default_rng(42)` 生成的**模拟因子数据**，
> 仅用于演示多因子回归的操作步骤和结果解读，**不代表真实 A 股因子结论**。
> 如需真实因子数据，请从 CSMAR、Wind 或 Tushare Pro 获取。

FF 三因子回归方程：

$$r_{i,t} - r_{f,t} = \alpha_i + \beta^{MKT}_i MKT_t
+ \beta^{SMB}_i SMB_t + \beta^{HML}_i HML_t + \varepsilon_{i,t}$$

In [ ]:
# Cell 7：生成示意因子数据（固定随机种子，可复现）
# 以下为教学示意数据，非真实 A 股因子

rng = np.random.default_rng(42)  # 固定种子
T   = len(rets)

mkt_sim = rng.normal(0.0002, 0.012, T)
smb_sim = rng.normal(0.0001, 0.008, T)
hml_sim = rng.normal(0.0001, 0.007, T)
umd_sim = rng.normal(0.0002, 0.009, T)

factors_sim = pd.DataFrame(
    {'MKT': mkt_sim, 'SMB': smb_sim, 'HML': hml_sim, 'UMD': umd_sim},
    index=rets.index,
)

print('以下均为教学示意数据（随机生成），非真实 A 股因子')
print()
ann_f  = factors_sim.mean() * TRADING_DAYS
ann_v  = factors_sim.std()  * np.sqrt(TRADING_DAYS)
sharpe = ann_f / ann_v
print(pd.DataFrame({'年化均值': ann_f, '年化波动': ann_v, '夏普': sharpe}).round(4))

In [ ]:
# Cell 8：FF 三因子回归（示意数据，等权组合为被解释变量）
# 以下为教学示意数据

y_port = excess_rets.mean(axis=1)   # 等权组合超额收益

X_ff3 = sm.add_constant(factors_sim[['MKT', 'SMB', 'HML']])
X_ff3.columns = ['alpha_const', 'MKT', 'SMB', 'HML']

result_ff3 = sm.OLS(y_port, X_ff3).fit()
print('=== FF 三因子回归结果（示意数据）===')
print(result_ff3.summary())
print()
print('提醒：因子数据为随机生成，系数仅演示解读方法，无实际意义')

In [ ]:
# Cell 9：各股票 FF 三因子回归汇总（示意数据）

ff3_rows = []
for ticker in TICKERS:
    res3 = sm.OLS(excess_rets[ticker], X_ff3).fit()
    ff3_rows.append({
        '股票': ticker,
        'a_年化': round(res3.params['alpha_const'] * TRADING_DAYS, 4),
        'a_t': round(res3.tvalues['alpha_const'], 3),
        'b_MKT': round(res3.params['MKT'], 4),
        'b_SMB': round(res3.params['SMB'], 4),
        'b_HML': round(res3.params['HML'], 4),
        'R2_FF3': round(res3.rsquared, 4),
        'R2_CAPM': round(capm_results[ticker].rsquared, 4),
        'R2提升': round(res3.rsquared - capm_results[ticker].rsquared, 4),
    })

print('=== 四只股票 FF 三因子回归汇总（示意数据）===')
print(pd.DataFrame(ff3_rows).set_index('股票').to_string())
print('注：实际 A 股数据中 FF3 通常比 CAPM 的 R2 高出 0.10~0.30')

## 7.6 Carhart 四因子：加入动量（示意数据）

> **以下仍为示意数据演示**，UMD 因子为随机生成，仅演示四因子回归的操作流程。

$$r_{i,t} - r_{f,t} = \alpha_i + \beta^{MKT} MKT_t
+ \beta^{SMB} SMB_t + \beta^{HML} HML_t + \beta^{UMD} UMD_t + \varepsilon_{i,t}$$

In [ ]:
# Cell 10：Carhart 四因子回归（示意数据）

X_c4 = sm.add_constant(factors_sim[['MKT', 'SMB', 'HML', 'UMD']])
X_c4.columns = ['alpha_const', 'MKT', 'SMB', 'HML', 'UMD']

c4_rows = []
for ticker in TICKERS:
    res4 = sm.OLS(excess_rets[ticker], X_c4).fit()
    c4_rows.append({
        '股票': ticker,
        'a_年化': round(res4.params['alpha_const'] * TRADING_DAYS, 4),
        'a_t': round(res4.tvalues['alpha_const'], 3),
        'a_p': round(res4.pvalues['alpha_const'], 4),
        'b_MKT': round(res4.params['MKT'], 4),
        'b_SMB': round(res4.params['SMB'], 4),
        'b_HML': round(res4.params['HML'], 4),
        'b_UMD': round(res4.params['UMD'], 4),
        'R2_C4': round(res4.rsquared, 4),
        'AIC': round(res4.aic, 2),
    })

print('=== 四只股票 Carhart 四因子回归汇总（示意数据）===')
print(pd.DataFrame(c4_rows).set_index('股票').to_string())
print()
print('alpha 检验：a_p < 0.05 说明在 5% 水平下存在显著超额 alpha')
print('因子数据为随机生成，以上数字仅演示操作，无实际意义')

## 7.7 因子相关性与多重共线性诊断

多因子回归的前提：各因子之间相关性不能过高，否则系数估计不稳定。

**方差膨胀因子 VIF**：
$$\text{VIF}_j = \frac{1}{1 - R_j^2}$$
其中 $R_j^2$ 是用其他所有因子回归第 $j$ 个因子的 $R^2$。
VIF < 5 正常；> 10 视为严重多重共线性。

In [ ]:
# Cell 11：因子相关性 + VIF 计算（示意数据）

factor_names = ['MKT', 'SMB', 'HML', 'UMD']
F = factors_sim[factor_names]

corr_matrix = F.corr()
print('=== 因子相关性矩阵（示意数据）===')
print(corr_matrix.round(4).to_string())
print()

def compute_vif(X_df):
    rows = []
    for col in X_df.columns:
        X_others = sm.add_constant(X_df.drop(columns=[col]))
        r2  = sm.OLS(X_df[col], X_others).fit().rsquared
        vif = 1 / (1 - r2) if r2 < 1 else float('inf')
        rows.append({'因子': col, 'R2': round(r2, 4), 'VIF': round(vif, 4)})
    return pd.DataFrame(rows).set_index('因子')

print('=== VIF（示意数据）===')
print(compute_vif(F).to_string())
print()
print('示意数据各因子独立生成，VIF 接近 1（低共线性）')
print('真实 A 股中 SMB 与 HML 可能存在较高相关性，需特别检验')

# 热图
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label='Pearson 相关系数')
ax.set_xticks(range(len(factor_names)))
ax.set_yticks(range(len(factor_names)))
ax.set_xticklabels(factor_names)
ax.set_yticklabels(factor_names)
ax.set_title('因子相关性热图（示意数据）')
for i in range(len(factor_names)):
    for j in range(len(factor_names)):
        val = corr_matrix.iloc[i, j]
        ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=11,
                color='white' if abs(val) > 0.5 else 'black')
plt.tight_layout()
plt.show()

## 7.8 业绩归因分解（CAPM，真实内置数据）

$$\underbrace{r_i - r_f}_{\text{总超额收益}}
= \underbrace{\alpha_i}_{\text{真实 alpha}}
+ \underbrace{\beta_i \cdot \overline{MKT}}_{\text{市场因子贡献}}$$

多因子版本将各因子贡献逐项拆解，便于判断收益来源于选股能力还是因子暴露。

In [ ]:
# Cell 12：业绩归因（CAPM 单因子，真实内置数据）

mkt_mean_ann = mkt_excess.mean() * TRADING_DAYS

attr_rows = []
for ticker in TICKERS:
    res       = capm_results[ticker]
    alpha_ann = res.params['alpha_const'] * TRADING_DAYS
    beta_val  = res.params['MKT']
    mkt_ctr   = beta_val * mkt_mean_ann
    total     = excess_rets[ticker].mean() * TRADING_DAYS
    attr_rows.append({
        '股票': ticker,
        '总超额收益年化': round(total, 4),
        'Alpha年化': round(alpha_ann, 4),
        '市场因子贡献': round(mkt_ctr, 4),
        '误差': round(total - alpha_ann - mkt_ctr, 8),
    })

df_attr = pd.DataFrame(attr_rows).set_index('股票')
print('=== CAPM 单因子业绩归因（年化，真实内置数据）===')
print(df_attr.round(4).to_string())
print(f'\n样本内年化市场超额收益（代理）：{mkt_mean_ann:.2%}')

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(TICKERS))
alpha_vals = [r['Alpha年化'] for r in attr_rows]
mkt_vals   = [r['市场因子贡献'] for r in attr_rows]
ax.bar(x, alpha_vals, 0.5, label='Alpha（选股贡献）',
       color='#2ca02c', alpha=0.85)
ax.bar(x, mkt_vals, 0.5, bottom=alpha_vals,
       label='市场因子贡献（Beta 暴露）', color='steelblue', alpha=0.85)
for i, total in enumerate(df_attr['总超额收益年化']):
    ax.text(i, total + 0.002, f'{total:.1%}', ha='center',
            fontsize=10, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(TICKERS)
ax.set_ylabel('年化超额收益')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title('CAPM 业绩归因：Alpha vs 市场因子贡献')
ax.axhline(0, color='black', lw=0.8)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 13：因子累计收益时序图

plot_data = factors_sim.copy()
plot_data['MKT'] = mkt_excess.values   # 替换为真实市场超额收益

factor_labels = [
    ('MKT', '市场超额收益（真实内置数据）'),
    ('SMB', 'SMB 规模因子（教学示意数据）'),
    ('HML', 'HML 价值因子（教学示意数据）'),
    ('UMD', 'UMD 动量因子（教学示意数据）'),
]
factor_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)
for ax, (fname, flabel), color in zip(axes, factor_labels, factor_colors):
    cumret  = (1 + plot_data[fname]).cumprod() - 1
    ann_ret = plot_data[fname].mean() * TRADING_DAYS
    ax.plot(plot_data.index, cumret, color=color, lw=1.2)
    ax.fill_between(plot_data.index, cumret, 0, color=color, alpha=0.15)
    ax.set_title(f'{flabel} | 年化 = {ann_ret:.2%}', fontsize=10)
    ax.set_ylabel('累计收益')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.axhline(0, color='gray', lw=0.8, linestyle=':')
axes[-1].set_xlabel('日期')
fig.suptitle('因子累计收益时序图', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7.9 本章小结

| 模型 | 因子数 | 用途 | 关键限制 |
|---|---|---|---|
| CAPM | 1（MKT） | beta 估计、基准收益 | 低 R2，忽略规模/价值/动量 |
| FF 三因子 | 3（+SMB, HML） | 风险归因、alpha 检验 | 忽略动量 |
| Carhart 四因子 | 4（+UMD） | 基金绩效评估 | 动量在 A 股有争议 |
| FF 五因子 | 5（+RMW, CMA） | 全面风险分解 | HML 冗余，未纳入动量 |

**A 股特殊注意**：壳价值、涨跌停、停牌、行业集中均会干扰因子构建，
推荐因子数据库：CSMAR、Wind、Tushare Pro。

## 习题参考解答

以下代码对应正文第 7.11 节习题，可直接运行。

In [ ]:
# === 习题 7.1：CAPM 代数计算 ===
print('习题 7.1：CAPM 预期收益与 alpha 计算')
rf_q1 = 0.02; rm_q1 = 0.08; beta_q1 = 1.4; actual_q1 = 0.12
er_q1    = rf_q1 + beta_q1 * (rm_q1 - rf_q1)
alpha_q1 = actual_q1 - er_q1
print(f'CAPM 预期收益 = {rf_q1:.1%} + {beta_q1} x {rm_q1-rf_q1:.1%} = {er_q1:.2%}')
print(f'实际收益 = {actual_q1:.1%}')
print(f'Alpha = {actual_q1:.1%} - {er_q1:.2%} = {alpha_q1:.2%}')
print('alpha > 0：在风险调整后仍有超额表现')

In [ ]:
# === 习题 7.2：四只股票 Beta 与预期收益 ===
print('习题 7.2：四只股票 Beta 与预期收益估计')
mkt_prem_ann = mkt_excess.mean() * TRADING_DAYS

ex72 = []
for ticker in TICKERS:
    res = capm_results[ticker]
    bv  = res.params['MKT']
    ex72.append({
        '股票': ticker,
        'Beta': round(bv, 4),
        'Alpha年化': round(res.params['alpha_const'] * TRADING_DAYS, 4),
        'CAPM预期年化': round(RF_ANNUAL + bv * mkt_prem_ann, 4),
        '实际超额年化': round(excess_rets[ticker].mean() * TRADING_DAYS, 4),
        'R2': round(res.rsquared, 4),
    })
df72 = pd.DataFrame(ex72).set_index('股票')
print(df72.round(4).to_string())
print(f'市场年化超额收益（代理）：{mkt_prem_ann:.2%}')
print(f'Beta 最高：{df72["Beta"].idxmax()}')
print(f'CAPM 预期收益最高：{df72["CAPM预期年化"].idxmax()}')

In [ ]:
# === 习题 7.3：R2 与系统性/特质风险 ===
print('习题 7.3：R2 与系统性/特质风险分析')
from fds import annualized_volatility
vol_vals = annualized_volatility(rets)

ex73 = []
for ticker in TICKERS:
    r2 = capm_results[ticker].rsquared
    ex73.append({
        '股票': ticker,
        '年化总波动': round(vol_vals[ticker], 4),
        'R2系统性占比': round(r2, 4),
        '1-R2特质占比': round(1 - r2, 4),
        '系统波动': round(vol_vals[ticker] * np.sqrt(r2), 4),
        '特质波动': round(vol_vals[ticker] * np.sqrt(1 - r2), 4),
    })
df73 = pd.DataFrame(ex73).set_index('股票')
print(df73.to_string())
mv = vol_vals.idxmax()
mr = df73['R2系统性占比'].idxmax()
print(f'\n波动率最高：{mv}，R2 最高：{mr}')
if mv != mr:
    print('结论：高波动率并不等于高 R2。高波动中若特质风险为主，则 R2 反而偏低。')

In [ ]:
# === 习题 7.4：多重共线性影响演示 ===
print('习题 7.4：多重共线性影响（构造高相关因子对比）')

rng2    = np.random.default_rng(99)
T2      = len(rets)
f_base  = rng2.normal(0, 0.01, T2)
f_high1 = f_base + rng2.normal(0, 0.003, T2)
f_high2 = f_base + rng2.normal(0, 0.003, T2)
f_indep = rng2.normal(0, 0.01, T2)
y_q4    = excess_rets['TECH'].values

X_low  = sm.add_constant(np.column_stack([f_high1, f_indep]))
X_high = sm.add_constant(np.column_stack([f_high1, f_high2]))
res_low  = sm.OLS(y_q4, X_low).fit()
res_high = sm.OLS(y_q4, X_high).fit()

print(f'高相关对（f1 vs f2）：Corr = {np.corrcoef(f_high1, f_high2)[0,1]:.4f}')
print(f'低相关对（f1 vs indep）：Corr = {np.corrcoef(f_high1, f_indep)[0,1]:.4f}')
print()
print('低共线性情况：')
print(f'  f1 t-stat = {res_low.tvalues[1]:.3f},  独立因子 t-stat = {res_low.tvalues[2]:.3f}')
print('高共线性情况：')
print(f'  f1 t-stat = {res_high.tvalues[1]:.3f},  f2 t-stat = {res_high.tvalues[2]:.3f}')
print()
print('结论：高共线性时 t-stat 均下降，各因子独立贡献难以区分。')
print('解决方案：(1) 正交化因子；(2) 删除冗余因子；(3) 岭回归')